<a href="https://colab.research.google.com/github/lavanyagkunder/ai_chatbot/blob/main/Copy_of_Untitled5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1.1: Upgrade pip and install required libraries
!pip install --upgrade pip
!pip install transformers accelerate gradio

# 1.2: (Optional) If you need GPU support for faster inference
import torch
print("GPU available:", torch.cuda.is_available())


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 44.1 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
GPU available: True


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

MODEL_NAME = "Qwen/Qwen3-4B"  # adjust if using a different repo

#2.1: Authenticate if private
from huggingface_hub import login
login() # <------------------------------------------------------- Create your own token at hugging face Token :)

# 2.2: Load tokenizer & model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype="auto",  # or float16 if supported
    trust_remote_code=True
)

# 2.3: Create a text-generation pipeline
llama_pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.7,
    top_p=0.9
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature', 'top_p'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [ ]:
# 1. Install dependencies
!pip install --upgrade googlesearch-python newspaper3k lxml_html_clean

# 2. Imports
from googlesearch import search
from newspaper import Article
import logging # Import the logging module here


# 3. New web_search implementation
def web_search(query, max_results=5, snippet_chars=300):
    """
    1) Runs a Google search for `query`, returns up to `max_results` URLs.
    2) Uses newspaper3k to download & parse each article.
    3) Returns list of dicts: [{'title','href','body'}]
    """
    results = []
    for url in search(query, num_results=max_results, lang="en"):
        try:
            art = Article(url)
            art.download()
            art.parse()
            text = art.text.replace("\n", " ").strip()
            snippet = text[:snippet_chars] + ("…" if len(text) > snippet_chars else "")
            results.append({
                "title": art.title or url,
                "href":  url,
                "body":  snippet
            })
        except Exception as e:
            logging.warning(f"Failed to parse {url}: {e}")
        if len(results) >= max_results:
            break
    return results

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 39.9 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for tinysegmenter: filename=tinysegmenter-0.3-py3-none-any.whl size=13634 sha256=065d428b14ed7034ce2674bb74ee6c11f3f3ac91b81507d6d30fafd2f61aee5b
  Stored in directory: /root/.cache/pip/wheels/a5/91/9f/00d66475960891a64867914273fcaf78df6cb04d905b104a2a
  Created wheel for feedfinder2: filename=feedfinder2-0.0.4-py3-none-any.whl size=3394 sha256=7f191382f606e135c2575

In [ ]:
def answer_with_citations(user_query):
    # 4.1: Run our improved web search (news → fallback text)
    hits = web_search(user_query, max_results=10)

    # 4.2: Build a prompt with numbered citations
    prompt_parts = [
        "You are an AI assistant. Use the following search results to answer the question below based on the best use of the information explain, list based on the question, and cite each source by the links\n"
    ]
    for idx, hit in enumerate(hits, start=1):
        title = hit.get("title", "").strip()
        body  = hit.get("body", "").strip()
        url   = hit.get("href", hit.get("url", "")).strip()
        prompt_parts.append(f"[{idx}] {title}\n{body}\nURL: {url}\n")
    prompt_parts.append(f"Question: {user_query}\nAnswer:")

    prompt = "\n".join(prompt_parts)

    # 4.3: Generate with LLaMA
    generated = llama_pipe(prompt, max_new_tokens=512, temperature=0.7)[0]["generated_text"]
    # Strip away the prompt to return just the answer portion:
    answer = generated[len(prompt):].strip()
    return answer


In [ ]:
import gradio as gr

# 1) Update your chat function to work with history state
def chat_fn(user_message, history):
    # Append user message
    history = history + [(user_message, None)]
    # Generate bot response
    bot_reply = answer_with_citations(user_message)
    # Replace the None with the actual reply
    history[-1] = (user_message, bot_reply)
    return history, history

# 2) Some simple custom CSS for bubble styling
css = """
/* Center the chat on the page */
.gradio-container { display: flex; justify-content: center; padding: 2rem; }
/* Chat container */
.chat-box { width: 100%; max-width: 700px; }
/* User bubbles */
.chatbot .message.user {
  background-color: #007AFF;
  color: white;
  border-radius: 16px 16px 0 16px;
  padding: 8px 12px;
  margin: 4px 0;
  align-self: flex-end;
  max-width: 80%;
}
/* Bot bubbles */
.chatbot .message.bot {
  background-color: #FFA500;
  color: #111;
  border-radius: 16px 16px 16px 0;
  padding: 8px 12px;
  margin: 4px 0;
  align-self: flex-start;
  max-width: 80%;
}
/* Input box styling */
input[type="text"] {
  border-radius: 20px;
  padding: 10px 16px;
  border: 1px solid #CCC;
}
"""

with gr.Blocks(css=css, title="📡 LLaMA 3.2 3B Web‑Search Agent") as demo:
    gr.Markdown("## Ask anything and get cited answers")
    with gr.Group(elem_classes="chat-box"):
        chatbot = gr.Chatbot(elem_classes="chatbot", height=500)
        state   = gr.State([])  # holds the list of (user,bot) tuples
        user_input = gr.Textbox(
            placeholder="Type your question and hit enter",
            show_label=False
        )
    # wire up: on submit, call chat_fn(user_input, state) → update chatbot & state
    user_input.submit(
        fn=chat_fn,
        inputs=[user_input, state],
        outputs=[chatbot, state]
    )

demo.launch(share=True)

/tmp/ipykernel_996/2035367703.py:47: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=css, title="📡 LLaMA 3.2 3B Web‑Search Agent") as demo:
/tmp/ipykernel_996/2035367703.py:50: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(elem_classes="chatbot", height=500)
/tmp/ipykernel_996/2035367703.py:50: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(elem_classes="chatbot", height=500)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a54a904eee0fd78587.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
